# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdsvr/flyrank/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

**Lane 4 — CTR / Engagement Opportunity Scoring.** Three signal tests on the lane slice (visible pages with position data: impressions ≥ 500, avg_position > 0, avg_position ≤ 20). At least one test is linked to a real FlyRank product flag.

Careful words: observed, measured, directional, decision-support.

In [1]:
import os
import pandas as pd
import numpy as np

# Anchor to repo root
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    os.chdir("../..")

df = pd.read_csv("data/processed/refresh_feature_vector.csv")
print(f"Full dataset: {len(df):,} rows, {df['client_id'].nunique()} clients")

Full dataset: 30,000 rows, 32 clients


## 1. Distributions

*Look before deciding: distributions of the key fields on my lane slice. Traffic metrics are heavy-tailed (a few giants, a long tail of tiny values), which changes everything below.*

In [2]:
# Lane 4 slice: visible pages with real position data
lane = df[
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
].copy()

print(f"Lane slice: {len(lane):,} rows, {lane['client_id'].nunique()} clients")
print(f"Declining rate (is_declining_label): {lane['is_declining_label'].mean():.3f}")
print()

print("ctr (%):")
print(lane["ctr"].describe().to_string())
print()
print("impressions_90d (heavy-tailed):")
print(lane["impressions_90d"].describe().to_string())
print()
print("avg_position:")
print(lane["avg_position"].describe().to_string())
print()

print("content_type distribution:")
print(lane["content_type"].value_counts().to_string())
print()
print("position_tier distribution:")
print(lane["position_tier"].value_counts().to_string())
print()
print("impression_tier distribution:")
print(lane["impression_tier"].value_counts().to_string())

Lane slice: 12,023 rows, 28 clients
Declining rate (is_declining_label): 0.599

ctr (%):
count    12023.000000
mean         0.312079
std          0.343159
min          0.000000
25%          0.100000
50%          0.210000
75%          0.410000
max          5.430000

impressions_90d (heavy-tailed):
count     12023.000000
mean       9880.425019
std       22738.394151
min         500.000000
25%        1366.500000
50%        3298.000000
75%        8873.000000
max      517715.000000

avg_position:
count    12023.000000
mean         9.269342
std          4.565431
min          0.200000
25%          5.700000
50%          8.200000
75%         12.400000
max         20.000000

content_type distribution:
content_type
keyword article       11842
feedly article          120
comparison article       61

position_tier distribution:
position_tier
page_1      7064
striking    4485
top_3        458
page_3_5      16

impression_tier distribution:
impression_tier
moderate     5717
good         5463
excellen

---
## 2. Signal test #1 — CTR-vs-Position (flag-linked)

**Claim:** *Pages with CTR below 0.5% at visible positions (1–20) are more likely to be declining than pages at or above the threshold.*

**Flag link:** FlyRank's `needs_ctr_fix` flag uses this exact logic — low CTR on a visible page triggers a CTR-review recommendation.

**Test:** Split the lane slice by `ctr < 0.5` vs `ctr ≥ 0.5`. Print n and declining rate per bucket.

In [3]:
lane["ctr_bucket"] = np.where(
    lane["ctr"] < 0.5, "ctr_under_0.5", "ctr_0.5_and_above"
)

s1 = (
    lane.groupby("ctr_bucket")
    .agg(
        n=("ctr", "count"),
        declining_rate=("is_declining_label", "mean"),
        mean_ctr=("ctr", "mean"),
        mean_position=("avg_position", "mean"),
    )
    .round(4)
)
print("Signal #1: CTR vs Position (bucket table)")
print(s1.to_string())
print()

rate_diff = s1.loc["ctr_under_0.5", "declining_rate"] - s1.loc["ctr_0.5_and_above", "declining_rate"]
print(f"Difference in declining rate: {rate_diff:.3f} (under_0.5 minus above)")
print()
print("Verdict: CONFIRMED — pages below 0.5% CTR show a materially higher declining rate (0.627 vs 0.475).")
print("The CTR-fix flag's core assumption holds in the data.")

Signal #1: CTR vs Position (bucket table)
                      n  declining_rate  mean_ctr  mean_position
ctr_bucket                                                      
ctr_0.5_and_above  2264          0.4753    0.8590         8.3491
ctr_under_0.5      9759          0.6271    0.1852         9.4828

Difference in declining rate: 0.152 (under_0.5 minus above)

Verdict: CONFIRMED — pages below 0.5% CTR show a materially higher declining rate (0.627 vs 0.475).
The CTR-fix flag's core assumption holds in the data.


---
## 2. Signal test #2 — Content-type CTR Gap

**Claim:** *Comparison articles capture systematically fewer clicks than keyword articles at the same search position.*

**Test:** Bucket table by `position_tier × content_type`. Show n and mean CTR per cell.

In [4]:
s2 = (
    lane.groupby(["position_tier", "content_type"])
    .agg(n=("ctr", "count"), mean_ctr=("ctr", "mean"))
    .round(4)
)
print("Signal #2: CTR by position_tier × content_type")
print(s2.to_string())
print()

print("Key comparison — page_1 only:")
p1 = lane[lane["position_tier"] == "page_1"]
p1_ct = p1.groupby("content_type")["ctr"].agg(["mean", "count"]).round(4)
print(p1_ct.to_string())
print()

print("Sample-size note: comparison article at page_1 has n=50 (adequate).")
print("striking + comparison has n=11 (below 30-threshold — treat as directional).")
print()
print("Verdict: CONFIRMED — on page 1, comparison articles average 0.08% CTR vs 0.34% for keyword articles.")

Signal #2: CTR by position_tier × content_type
                                     n  mean_ctr
position_tier content_type                      
page_1        comparison article    50    0.0764
              feedly article        83    0.4335
              keyword article     6931    0.3396
page_3_5      keyword article       16    0.2169
striking      comparison article    11    0.1400
              feedly article        34    0.3585
              keyword article     4440    0.2664
top_3         feedly article         3    2.6267
              keyword article      455    0.3315

Key comparison — page_1 only:
                      mean  count
content_type                     
comparison article  0.0764     50
feedly article      0.4335     83
keyword article     0.3396   6931

Sample-size note: comparison article at page_1 has n=50 (adequate).
striking + comparison has n=11 (below 30-threshold — treat as directional).

Verdict: CONFIRMED — on page 1, comparison articles average 0.08% C

---
## 2. Signal test #3 — Volume Leverage

**Claim:** *Pages with higher impressions have more leverage — a CTR fix on a high-traffic page yields more absolute extra clicks.*

**Test:** Bucket by `impression_tier`. Show n, mean CTR, declining rate, and median impressions per tier.

In [5]:
s3 = (
    lane.groupby("impression_tier")
    .agg(
        n=("ctr", "count"),
        mean_ctr=("ctr", "mean"),
        declining_rate=("is_declining_label", "mean"),
        median_impressions=("impressions_90d", "median"),
    )
    .round(4)
)
print("Signal #3: CTR by impression_tier")
print(s3.to_string())
print()

print("Interpretation: moderate-tier pages show the highest declining rate (0.666) and the lowest mean CTR (0.267).")
print("But excellent-tier pages have >10x the median impressions — even a tiny CTR improvement there")
print("yields far more absolute clicks. Volume leverage is real.")
print()
print("Verdict: CONFIRMED — higher-impression tiers have lower declining rates but vastly more traffic leverage.")

Signal #3: CTR by impression_tier
                    n  mean_ctr  declining_rate  median_impressions
impression_tier                                                    
excellent         843    0.3580          0.4211             49350.0
good             5463    0.3519          0.5550              7175.0
moderate         5717    0.2672          0.6663              1305.0

Interpretation: moderate-tier pages show the highest declining rate (0.666) and the lowest mean CTR (0.267).
But excellent-tier pages have >10x the median impressions — even a tiny CTR improvement there
yields far more absolute clicks. Volume leverage is real.

Verdict: CONFIRMED — higher-impression tiers have lower declining rates but vastly more traffic leverage.


---
## 3. The flag-linked test — CTR threshold, deeper

*Signal #1 showed the 0.5% threshold holds overall. But does the threshold behave the same across all position tiers? A single threshold that works for top-of-page-1 may be wrong for striking positions.*

In [6]:
# Deeper dive: does the 0.5% threshold work equally at every position tier?
flag_deep = (
    lane.groupby(["position_tier", "ctr_bucket"])
    .agg(
        n=("ctr", "count"),
        declining_rate=("is_declining_label", "mean"),
        mean_ctr=("ctr", "mean"),
    )
    .round(4)
)
print("Flag-linked deep dive: CTR < 0.5 by position_tier")
print(flag_deep.to_string())
print()

print("Observations:")
print("- top_3: under-0.5 declining rate is 0.858 vs 0.386 above — a 47pp gap. The threshold bites hard here.")
print("- page_1: 0.612 vs 0.456 — a 16pp gap. Meaningful but smaller.")
print("- striking: 0.629 vs 0.533 — a 10pp gap. Still directional.")
print("- page_3_5: only 16 rows; insufficient data for a verdict.")
print()
print("Takeaway: the 0.5 threshold works across tiers but is most discriminative at top_3.")
print("A position-aware threshold (e.g., 0.3 for top_3, 0.5 for page_1, 0.7 for striking)")
print("would likely sharpen precision. This is exactly why a learned ranking can beat a fixed rule.")

Flag-linked deep dive: CTR < 0.5 by position_tier
                                    n  declining_rate  mean_ctr
position_tier ctr_bucket                                       
page_1        ctr_0.5_and_above  1476          0.4560    0.8603
              ctr_under_0.5      5588          0.6117    0.2011
page_3_5      ctr_0.5_and_above     1          0.0000    0.9800
              ctr_under_0.5        15          0.5333    0.1660
striking      ctr_0.5_and_above   673          0.5334    0.8501
              ctr_under_0.5      3812          0.6293    0.1638
top_3         ctr_0.5_and_above   114          0.3860    0.8948
              ctr_under_0.5       344          0.8576    0.1649

Observations:
- top_3: under-0.5 declining rate is 0.858 vs 0.386 above — a 47pp gap. The threshold bites hard here.
- page_1: 0.612 vs 0.456 — a 16pp gap. Meaningful but smaller.
- striking: 0.629 vs 0.533 — a 10pp gap. Still directional.
- page_3_5: only 16 rows; insufficient data for a verdict.

Takeaway:

---
## 4. What this means in practice

*Two or three sentences for a content team.*

In [7]:
print("What a content team should take from this:")
print()
print("1. The CTR-fix flag's core assumption holds: visible pages with CTR below 0.5% are genuinely more")
print("   likely to be declining — the flag points at real problems.")
print("2. Comparison articles on page 1 capture ~4x fewer clicks than keyword articles at the same")
print("   position (0.08% vs 0.34% mean CTR). A separate CTR baseline for comparison articles would")
print("   surface more of them for review.")
print("3. The 0.5% threshold is most discriminative at top_3 positions (47pp gap) and least at striking")
print("   positions (10pp gap). A position-aware rule should not use one threshold for all tiers.")

What a content team should take from this:

1. The CTR-fix flag's core assumption holds: visible pages with CTR below 0.5% are genuinely more
   likely to be declining — the flag points at real problems.
2. Comparison articles on page 1 capture ~4x fewer clicks than keyword articles at the same
   position (0.08% vs 0.34% mean CTR). A separate CTR baseline for comparison articles would
   surface more of them for review.
3. The 0.5% threshold is most discriminative at top_3 positions (47pp gap) and least at striking
   positions (10pp gap). A position-aware rule should not use one threshold for all tiers.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.